# Whole-session locomotor spectral analysis

This notebook runs the first-stage, whole-session analysis across all `behavior_v1.h5` files represented in `cc_data`.

It expects the batch module at:

```text
src/proc/whole_session_spectral_batch.py
```

The workflow produces:

- one row per animal-session,
- long-form whole-session PSDs,
- quality-control and error tables,
- longitudinal plots,
- phase summaries,
- exploratory mixed-effects screening.

The main goal is to identify which spectral outcomes should be carried forward into within-session, trial-level, and event-aligned analyses.


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display


## 1. Locate the repository and import local modules

In [ ]:
def find_repo_root(start_path=None):
    start_path = Path.cwd() if start_path is None else Path(start_path)
    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src").is_dir():
            return candidate

    raise RuntimeError(
        "Could not find the repository root containing the src directory."
    )


repo_root = find_repo_root()

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repository root:", repo_root)


In [ ]:
import src.utils.config as config
import src.utils.data_io as dio

from src.proc.whole_session_spectral_batch import (
    WholeSessionSpectralConfig,
    build_whole_session_spectral_tables,
    save_whole_session_spectral_tables,
)

print("RAW_BASE:", config.RAW_BASE)
print("PROC_BASE:", config.PROC_BASE)


## 2. Build `cc_data`

In [ ]:
data_root = config.RAW_BASE
pdata_root = config.PROC_BASE

cc_data = dio.build_classical_conditioning_dict(data_root)

print("Top-level animal entries:", len(cc_data))


## 3. Configure the analysis

Primary signal-processing path:

```text
native dist_net_cm
→ 20-ms Savitzky–Golay derivative
→ zero-phase 20-Hz low-pass
→ 100-Hz analysis series
→ 10-s Welch PSD with 50% overlap
```


In [ ]:
spectral_cfg = WholeSessionSpectralConfig(
    analysis_hz=100.0,

    derivative_window_ms=20.0,
    derivative_polyorder=3,

    speed_lowpass_hz=20.0,
    filter_order=4,

    welch_window_s=10.0,
    welch_overlap_fraction=0.50,

    spectral_fmin_hz=0.10,
    spectral_fmax_hz=20.0,

    behavior_window_s=2.0,
    movement_threshold_cms=0.20,

    spectral_slope_fmin_hz=1.0,
    spectral_slope_fmax_hz=10.0,
)

spectral_cfg


## 4. Select sessions

Leave these as `None` to process every session represented in `cc_data`.

Examples:

```python
animals_to_process = ["NML_06"]
phases_to_process = ["repeated_exposure", "air_training"]
```


In [ ]:
animals_to_process = None
phases_to_process = None


## 5. Run the whole-session batch analysis

In [ ]:
(
    whole_session_df,
    whole_session_psd_df,
    whole_session_errors_df,
) = build_whole_session_spectral_tables(
    cc_data=cc_data,
    config=spectral_cfg,

    # Leave as None when pdata_io already uses config.PROC_BASE.
    root=None,

    animals=animals_to_process,
    phases=phases_to_process,
    verbose=True,
)

print()
print("Sessions processed:", len(whole_session_df))
print("PSD rows:", len(whole_session_psd_df))
print("Sessions skipped:", len(whole_session_errors_df))


## 6. Save the output tables

In [ ]:
whole_session_output_dir = (
    Path(config.PROC_BASE)
    / "spectral_analysis"
    / "whole_session"
)

saved_paths = save_whole_session_spectral_tables(
    whole_session_df=whole_session_df,
    whole_session_psd_df=whole_session_psd_df,
    errors_df=whole_session_errors_df,
    output_dir=whole_session_output_dir,
)

for name, path in saved_paths.items():
    print(f"{name}: {path}")


## 7. Processing coverage and errors

Confirm that the expected animals, phases, and dates were included before interpreting any results.


In [ ]:
if whole_session_df.empty:
    raise RuntimeError(
        "No sessions were processed. Inspect whole_session_errors_df."
    )

coverage = (
    whole_session_df
    .groupby(["animal", "phase"], dropna=False)
    .agg(
        sessions=("date", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
)

display(coverage)

if whole_session_errors_df.empty:
    print("No processing errors.")
else:
    display(whole_session_errors_df)


## 8. Main whole-session feature table

In [ ]:
overview_columns = [
    "animal",
    "date",
    "phase",
    "animal_session",
    "phase_session",
    "phase_n_sessions",
    "phase_progress_0_1",

    "duration_min",
    "mean_abs_speed_cms",
    "fraction_moving_2s_windows",
    "net_displacement_cm",
    "path_distance_from_native_speed_cm",

    "signed_total_power_0p1_20",
    "signed_relative_power_slow_0p5_2",
    "signed_relative_power_intermediate_2_5",
    "signed_dominant_frequency_hz",
    "signed_spectral_centroid_hz",
    "signed_spectral_entropy",
    "signed_spectral_slope_1_10",

    "path_relative_power_slow_0p5_2",
    "path_relative_power_intermediate_2_5",
    "path_spectral_centroid_hz",
    "path_spectral_entropy",

    "near_immobile_2s_mean_abs_speed_p99_cms",
    "native_vs_stored_speed_r",
]

available_overview_columns = [
    column
    for column in overview_columns
    if column in whole_session_df.columns
]

display(
    whole_session_df[
        available_overview_columns
    ].sort_values(["animal", "animal_session"])
)


## 9. Quality-control summaries

In [ ]:
quality_columns = [
    "animal",
    "date",
    "phase",
    "duration_min",
    "original_sampling_hz",
    "derivative_window_ms_actual",
    "near_immobile_2s_windows",
    "near_immobile_2s_mean_abs_speed_p99_cms",
    "native_vs_stored_speed_r",
    "native_vs_stored_speed_rmse_cms",
]

available_quality_columns = [
    column
    for column in quality_columns
    if column in whole_session_df.columns
]

display(
    whole_session_df[
        available_quality_columns
    ].sort_values(["animal", "animal_session"])
)

print(
    "Configured 2-s movement threshold:",
    spectral_cfg.movement_threshold_cms,
    "cm/s",
)


## 10. Descriptive summaries by phase

In [ ]:
features_to_summarize = [
    "mean_abs_speed_cms",
    "fraction_moving_2s_windows",

    "signed_total_power_0p1_20",
    "signed_relative_power_slow_0p5_2",
    "signed_relative_power_intermediate_2_5",
    "signed_dominant_frequency_hz",
    "signed_spectral_centroid_hz",
    "signed_spectral_entropy",

    "path_relative_power_slow_0p5_2",
    "path_relative_power_intermediate_2_5",
    "path_spectral_centroid_hz",
    "path_spectral_entropy",
]

available_summary_features = [
    column
    for column in features_to_summarize
    if column in whole_session_df.columns
]

phase_summary = (
    whole_session_df
    .groupby("phase")[available_summary_features]
    .agg(["count", "mean", "median", "std"])
)

display(phase_summary)


## 11. Longitudinal plotting helper

In [ ]:
def plot_longitudinal_session_feature(
    session_df,
    feature,
    ylabel=None,
    title=None,
):
    if feature not in session_df.columns:
        raise KeyError(f"Feature not found: {feature}")

    fig, ax = plt.subplots(figsize=(11, 5))

    for animal, animal_df in session_df.groupby("animal"):
        animal_df = animal_df.sort_values("animal_session")

        ax.plot(
            animal_df["animal_session"],
            animal_df[feature],
            marker="o",
            label=animal,
        )

    ax.set_xlabel("Sequential session within animal")
    ax.set_ylabel(ylabel if ylabel is not None else feature)
    ax.set_title(title if title is not None else feature)

    ax.legend(
        title="Animal",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    fig.tight_layout()

    return fig, ax


### Conventional whole-session behavior

In [ ]:
plot_longitudinal_session_feature(
    whole_session_df,
    feature="fraction_moving_2s_windows",
    ylabel="Fraction of 2-s windows classified as moving",
    title="Whole-session movement fraction",
)

plt.show()


In [ ]:
plot_longitudinal_session_feature(
    whole_session_df,
    feature="mean_abs_speed_cms",
    ylabel="Mean absolute speed (cm/s)",
    title="Whole-session mean absolute speed",
)

plt.show()


### Primary whole-session spectral outcomes

In [ ]:
plot_longitudinal_session_feature(
    whole_session_df,
    feature="signed_relative_power_slow_0p5_2",
    ylabel="Relative signed-speed power, 0.5–2 Hz",
    title="Whole-session relative power: 0.5–2 Hz",
)

plt.show()


In [ ]:
plot_longitudinal_session_feature(
    whole_session_df,
    feature="signed_relative_power_intermediate_2_5",
    ylabel="Relative signed-speed power, 2–5 Hz",
    title="Whole-session relative power: 2–5 Hz",
)

plt.show()


In [ ]:
plot_longitudinal_session_feature(
    whole_session_df,
    feature="signed_spectral_entropy",
    ylabel="Signed-speed spectral entropy",
    title="Whole-session spectral entropy",
)

plt.show()


In [ ]:
plot_longitudinal_session_feature(
    whole_session_df,
    feature="signed_spectral_centroid_hz",
    ylabel="Signed-speed spectral centroid (Hz)",
    title="Whole-session spectral centroid",
)

plt.show()


## 12. Complete PSD curves by phase

These are descriptive medians across animal-session PSDs at each frequency.


In [ ]:
def plot_phase_psd(
    psd_df,
    signal="signed_speed",
    relative=False,
):
    value_column = (
        "relative_power_density"
        if relative
        else "power"
    )

    plot_df = psd_df[
        psd_df["signal"] == signal
    ].copy()

    phase_psd = (
        plot_df
        .groupby(
            ["phase", "frequency_hz"],
            as_index=False,
        )[value_column]
        .median()
    )

    fig, ax = plt.subplots(figsize=(10, 5))

    for phase, phase_df in phase_psd.groupby("phase"):
        if relative:
            ax.plot(
                phase_df["frequency_hz"],
                phase_df[value_column],
                label=phase,
            )
        else:
            ax.semilogy(
                phase_df["frequency_hz"],
                phase_df[value_column],
                label=phase,
            )

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel(
        "Median relative power density"
        if relative
        else "Median session PSD [(cm/s)²/Hz]"
    )
    ax.set_title(f"Whole-session PSD by phase: {signal}")
    ax.legend()
    fig.tight_layout()

    return fig, ax


In [ ]:
plot_phase_psd(
    whole_session_psd_df,
    signal="signed_speed",
    relative=False,
)

plt.show()


In [ ]:
plot_phase_psd(
    whole_session_psd_df,
    signal="signed_speed",
    relative=True,
)

plt.show()


In [ ]:
plot_phase_psd(
    whole_session_psd_df,
    signal="path_speed",
    relative=False,
)

plt.show()


## 13. Correlations with conventional behavior

Total power is expected to track movement amount. Relative power, centroid, entropy, dominant frequency, and slope are the more important candidates for information beyond amplitude.


In [ ]:
correlation_columns = [
    "mean_abs_speed_cms",
    "fraction_moving_2s_windows",
    "path_distance_from_native_speed_cm",

    "signed_total_power_0p1_20",
    "signed_relative_power_slow_0p5_2",
    "signed_relative_power_intermediate_2_5",
    "signed_dominant_frequency_hz",
    "signed_spectral_centroid_hz",
    "signed_spectral_entropy",

    "path_relative_power_slow_0p5_2",
    "path_relative_power_intermediate_2_5",
    "path_spectral_centroid_hz",
    "path_spectral_entropy",
]

available_correlation_columns = [
    column
    for column in correlation_columns
    if column in whole_session_df.columns
]

session_correlations = (
    whole_session_df[
        available_correlation_columns
    ]
    .corr()
)

display(session_correlations.round(3))


## 14. Scatterplots against mean speed

In [ ]:
def plot_feature_against_speed(
    session_df,
    feature,
    speed_feature="mean_abs_speed_cms",
):
    fig, ax = plt.subplots(figsize=(7, 5))

    for animal, animal_df in session_df.groupby("animal"):
        ax.scatter(
            animal_df[speed_feature],
            animal_df[feature],
            label=animal,
        )

    ax.set_xlabel(speed_feature)
    ax.set_ylabel(feature)
    ax.set_title(f"{feature} versus {speed_feature}")

    ax.legend(
        title="Animal",
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )

    fig.tight_layout()

    return fig, ax


In [ ]:
plot_feature_against_speed(
    whole_session_df,
    feature="signed_relative_power_slow_0p5_2",
)

plt.show()


In [ ]:
plot_feature_against_speed(
    whole_session_df,
    feature="signed_relative_power_intermediate_2_5",
)

plt.show()


In [ ]:
plot_feature_against_speed(
    whole_session_df,
    feature="signed_spectral_entropy",
)

plt.show()


## 15. Optional mixed-effects screening

Each row is an animal-session, with a random intercept for animal.

These are screening models rather than final inferential models. With only five animals, keep the fixed-effect structure parsimonious and inspect every animal trajectory.


In [ ]:
import statsmodels.formula.api as smf


def fit_whole_session_mixed_model(
    session_df,
    outcome,
):
    required_columns = [
        outcome,
        "animal",
        "phase",
        "phase_progress_0_1",
        "mean_abs_speed_cms",
        "fraction_moving_2s_windows",
    ]

    model_df = (
        session_df[
            required_columns
        ]
        .dropna()
        .copy()
    )

    if model_df["animal"].nunique() < 2:
        raise ValueError(
            "At least two animals are required for a mixed model."
        )

    model_df["mean_abs_speed_c"] = (
        model_df["mean_abs_speed_cms"]
        - model_df["mean_abs_speed_cms"].mean()
    )

    model_df["fraction_moving_c"] = (
        model_df["fraction_moving_2s_windows"]
        - model_df["fraction_moving_2s_windows"].mean()
    )

    formula = (
        f"{outcome} ~ "
        "C(phase) + "
        "phase_progress_0_1 + "
        "mean_abs_speed_c + "
        "fraction_moving_c"
    )

    model = smf.mixedlm(
        formula=formula,
        data=model_df,
        groups=model_df["animal"],
        re_formula="1",
    )

    result = model.fit(
        reml=True,
        method="lbfgs",
    )

    return result, model_df


In [ ]:
candidate_outcomes = [
    "signed_relative_power_slow_0p5_2",
    "signed_relative_power_intermediate_2_5",
    "signed_dominant_frequency_hz",
    "signed_spectral_centroid_hz",
    "signed_spectral_entropy",
]

mixed_model_results = {}

for outcome in candidate_outcomes:
    try:
        result, model_data = fit_whole_session_mixed_model(
            whole_session_df,
            outcome=outcome,
        )

        mixed_model_results[outcome] = result

        print()
        print("=" * 80)
        print(outcome)
        print("=" * 80)
        print(result.summary())

    except Exception as exc:
        print()
        print(
            f"Could not fit {outcome}: "
            f"{type(exc).__name__}: {exc}"
        )


## 16. Interpretation sequence

Review the outputs in this order:

1. **Coverage and quality control**  
   Confirm expected animals and sessions, low immobility noise, and strong agreement between native-derived and stored speed.

2. **Conventional behavior**  
   Inspect movement fraction, mean speed, distance, and longitudinal phase progression.

3. **Absolute spectral power**  
   Determine how strongly total spectral power tracks movement amount.

4. **Relative spectral shape**  
   Focus on 0.5–2 Hz, 2–5 Hz, dominant frequency, centroid, entropy, and slope.

5. **Signed versus path speed**  
   Signed-speed-only effects may reflect directional reversals. Effects present in both may reflect general locomotor modulation.

6. **Animal-level reproducibility**  
   Look for consistent effect directions across animals, not merely session-level significance.

7. **Select outcomes for the next stage**  
   Carry reproducible features forward into:
   - within-session early/middle/late analysis,
   - event-aligned air, LED, and tone analyses,
   - intact locomotor-bout analysis,
   - speed-matched comparisons.

Do not concatenate separated moving samples into one movement-only series, because the artificial joins create spectral transients.


## 17. Reload saved tables without rerunning preprocessing

In [ ]:
# whole_session_df = pd.read_csv(
#     whole_session_output_dir
#     / "whole_session_spectral_features.csv"
# )
#
# whole_session_psd_df = pd.read_csv(
#     whole_session_output_dir
#     / "whole_session_psd_long.csv"
# )
#
# whole_session_errors_df = pd.read_csv(
#     whole_session_output_dir
#     / "whole_session_spectral_errors.csv"
# )
